# ArbitrAgent — Curriculum-Trained Negotiation Agent

In [ ]:
!pip install -q openenv transformers trl datasets sentence-transformers diplomacy torch matplotlib

In [ ]:
# Clone repo and set paths (replace with your repo URL)
import os
import sys
import subprocess
REPO_URL = "https://github.com/your-username/Play-gent.git"  # or arbitragent
REPO_NAME = "Play-gent"  # folder name after clone
if not os.path.exists("envs/diplomacy_env.py"):  # not already in repo
    subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
ROOT = os.getcwd()
sys.path.insert(0, ROOT)
print("ROOT:", ROOT)

In [ ]:
# Load DiplomacyNegotiationEnv, run reset() and render()
from envs.diplomacy_env import DiplomacyNegotiationEnv

env = DiplomacyNegotiationEnv(power_name="ENGLAND", seed=42)
obs, info = env.reset()
print("Observation shape:", obs.shape)
print("Info:", info)
print()
env.render()

In [ ]:
# Load reward model, score 4 different moves
import torch
from transformers import AutoTokenizer
from reward_model import DiplomacyRewardModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rm_path = "reward_model.pt"
if not os.path.exists(rm_path):
    rm_path = "training/checkpoints/reward_model.pt"

tokenizer_rm = AutoTokenizer.from_pretrained("distilbert-base-uncased")
reward_model = DiplomacyRewardModel().to(device)
if os.path.exists(rm_path):
    reward_model.load_state_dict(torch.load(rm_path, map_location=device))
    reward_model.eval()
else:
    print("Warning: reward_model.pt not found; using untrained weights (scores will be random).")

state_text = "DIPLOMACY GAME STATE\nPhase: F1901M\nPlaying as: ENGLAND. My units: Fleet LON, Fleet EDI. My supply centers: LON, EDI (2 centers). Other powers: FRANCE, GERMANY, RUSSIA, ..."
moves = [
    "I will support France into Belgium and move my fleet to North Sea.",
    "Hold both fleets and open negotiations with Germany.",
    "Attack France immediately with both units.",
    "Random gibberish xyz hold.",
]
scores = [reward_model.score(state_text, m, tokenizer_rm, device) for m in moves]
for m, s in zip(moves, scores):
    print(f"Score: {s:.4f}  |  {m[:60]}...")
print("\nReward model loaded and 4 moves scored.")

In [ ]:
# Abbreviated Phase 1 GRPO — 20 steps, plot reward curve
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoTokenizer

PHASE1_STEPS = 20
PHASE1_OUTPUT = "grpo_phase1_colab"

# Build prompts from env (no large JSON needed)
from envs.diplomacy_env import DiplomacyNegotiationEnv
env = DiplomacyNegotiationEnv(seed=42)
prompts_list = []
for _ in range(80):
    env.reset()
    prompts_list.append(env._get_state_text())

dataset = Dataset.from_list([{"prompt": p} for p in prompts_list])
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

def _extract_completion_text(c):
    if isinstance(c, str):
        return c.strip()
    if isinstance(c, list) and c:
        last = c[-1]
        if isinstance(last, dict) and "content" in last:
            return last["content"].strip()
    return ""

def make_phase1_reward(reward_model, tokenizer_rm, device):
    def fn(completions, prompts=None, **kwargs):
        if prompts is None:
            prompts = [""] * len(completions)
        texts = [_extract_completion_text(c) for c in completions]
        return [reward_model.score(s, a, tokenizer_rm, device) for s, a in zip(prompts, texts)]
    return fn

config = GRPOConfig(
    output_dir=PHASE1_OUTPUT,
    max_steps=PHASE1_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=2,
    save_steps=PHASE1_STEPS,
    report_to="none",
    max_completion_length=80,
    num_generations=4,
)

phase1_reward_log = []
class Phase1Callback:
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "reward" in str(logs):
            for k, v in (logs or {}).items():
                if "reward" in k.lower() and isinstance(v, (int, float)):
                    phase1_reward_log.append(float(v))
                    break

trainer_p1 = GRPOTrainer(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    args=config,
    reward_funcs=make_phase1_reward(reward_model, tokenizer_rm, device),
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer_p1.add_callback(Phase1Callback())
trainer_p1.train()
trainer_p1.save_model(PHASE1_OUTPUT)
tokenizer.save_pretrained(PHASE1_OUTPUT)
if not phase1_reward_log and hasattr(trainer_p1, 'state') and trainer_p1.state.log_history:
    for entry in trainer_p1.state.log_history:
        if isinstance(entry.get("reward"), (int, float)):
            phase1_reward_log.append(float(entry["reward"]))

if phase1_reward_log:
    plt.figure(figsize=(10, 4))
    plt.plot(phase1_reward_log, alpha=0.6, label="Step reward")
    w = min(5, len(phase1_reward_log))
    ma = np.convolve(phase1_reward_log, np.ones(w)/w, mode="valid")
    plt.plot(range(w-1, len(phase1_reward_log)), ma, linewidth=2, label="Moving avg")
    plt.xlabel("Step"); plt.ylabel("Reward"); plt.title("Phase 1 GRPO (Diplomacy) Reward Curve"); plt.legend(); plt.tight_layout(); plt.show()
else:
    plt.figure(figsize=(6, 3)); plt.text(0.5, 0.5, "Phase 1 complete (no reward log)", ha="center"); plt.axis("off"); plt.show()

In [ ]:
# Load HumanImitationEnv, run reset() and render()
import json

# Ensure minimal Phase 2 data exists (from Diplomacy env if no JSON)
data_path = "training/data/selfplay_states.json"
if not os.path.exists(data_path):
    os.makedirs("training/data", exist_ok=True)
    from envs.diplomacy_env import DiplomacyNegotiationEnv
    env = DiplomacyNegotiationEnv(seed=42)
    fallback = []
    for i in range(100):
        env.reset()
        fallback.append({
            "game_id": str(i), "phase": "F1901M", "power": "ENGLAND",
            "state_text": env._get_state_text(), "reward": 0.0, "sc_count": 3, "sc_delta": 0,
            "is_winner": False, "is_eliminated": False,
        })
    with open(data_path, "w") as f:
        json.dump(fallback, f)
    print("Created fallback training/data/selfplay_states.json")

from envs.human_imitation_env import HumanImitationEnv
env2 = HumanImitationEnv(data_path=data_path, seed=42)
obs2, info2 = env2.reset()
print("Observation shape:", obs2.shape)
print("Info:", info2)
print()
env2.render()

In [ ]:
# Abbreviated Phase 2 GRPO — 10 steps continuing from Phase 1, plot reward curve
with open(data_path) as f:
    states_p2 = json.load(f)
sample_p2 = list(np.random.choice(states_p2, size=min(200, len(states_p2)), replace=False))
dataset_p2 = Dataset.from_list([{"prompt": s["state_text"]} for s in sample_p2])

def compute_reward_p2(completions, prompts=None, **kwargs):
    rewards = []
    for c in completions:
        text = _extract_completion_text(c).lower()
        r = 0.0
        if any(w in text for w in ["ally", "alliance", "coalition", "support"]): r += 0.3
        if any(w in text for w in ["attack", "advance", "take", "capture"]): r += 0.2
        if any(w in text for w in ["defend", "protect", "hold", "guard"]): r += 0.2
        if any(w in text for w in ["because", "therefore", "since", "strategic"]): r += 0.2
        if any(w in text for w in ["bluff", "pressure", "leverage", "signal"]): r += 0.1
        rewards.append(r)
    return rewards

PHASE2_STEPS = 10
PHASE2_OUTPUT = "training/checkpoints/phase2_colab"
os.makedirs(PHASE2_OUTPUT, exist_ok=True)

config_p2 = GRPOConfig(
    output_dir=PHASE2_OUTPUT,
    max_steps=PHASE2_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=2,
    save_steps=PHASE2_STEPS,
    report_to="none",
    max_completion_length=80,
    num_generations=4,
)

phase2_reward_log = []
class Phase2Callback:
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            for k, v in (logs or {}).items():
                if "reward" in k.lower() and isinstance(v, (int, float)):
                    phase2_reward_log.append(float(v))
                    break

trainer_p2 = GRPOTrainer(
    model=PHASE1_OUTPUT,
    args=config_p2,
    reward_funcs=compute_reward_p2,
    train_dataset=dataset_p2,
    processing_class=tokenizer,
)
trainer_p2.add_callback(Phase2Callback())
trainer_p2.train()
trainer_p2.save_model(PHASE2_OUTPUT)
tokenizer.save_pretrained(PHASE2_OUTPUT)
if not phase2_reward_log and hasattr(trainer_p2, 'state') and trainer_p2.state.log_history:
    for entry in trainer_p2.state.log_history:
        if isinstance(entry.get("reward"), (int, float)):
            phase2_reward_log.append(float(entry["reward"]))

if phase2_reward_log:
    plt.figure(figsize=(10, 4))
    plt.plot(phase2_reward_log, alpha=0.6, label="Step reward")
    w = min(5, len(phase2_reward_log))
    ma = np.convolve(phase2_reward_log, np.ones(w)/w, mode="valid")
    plt.plot(range(w-1, len(phase2_reward_log)), ma, linewidth=2, label="Moving avg")
    plt.xlabel("Step"); plt.ylabel("Reward"); plt.title("Phase 2 GRPO (Human Imitation) Reward Curve"); plt.legend(); plt.tight_layout(); plt.show()
else:
    plt.figure(figsize=(6, 3)); plt.text(0.5, 0.5, "Phase 2 complete (no reward log)", ha="center"); plt.axis("off"); plt.show()

In [ ]:
# Plot both curves side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
if phase1_reward_log:
    ax1.plot(phase1_reward_log, alpha=0.6)
    w = min(5, len(phase1_reward_log))
    ma = np.convolve(phase1_reward_log, np.ones(w)/w, mode="valid")
    ax1.plot(range(w-1, len(phase1_reward_log)), ma, linewidth=2)
ax1.set_xlabel("Step"); ax1.set_ylabel("Reward"); ax1.set_title("Phase 1 (Diplomacy)")
if phase2_reward_log:
    ax2.plot(phase2_reward_log, alpha=0.6)
    w = min(5, len(phase2_reward_log))
    ma = np.convolve(phase2_reward_log, np.ones(w)/w, mode="valid")
    ax2.plot(range(w-1, len(phase2_reward_log)), ma, linewidth=2)
ax2.set_xlabel("Step"); ax2.set_ylabel("Reward"); ax2.set_title("Phase 2 (Human Imitation)")
plt.suptitle("ArbitrAgent Curriculum GRPO Reward Curves"); plt.tight_layout(); plt.show()

In [ ]:
# Side-by-side inference: base TinyLlama vs trained model on same negotiation state
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

negotiation_state = "DIPLOMACY GAME STATE\nPhase: F1902M\nPlaying as: ENGLAND. My units: Fleet LON, Fleet NTH, Army LVP. My supply centers: LON, EDI, LVP (3 centers). Other powers: FRANCE (4), GERMANY (3), RUSSIA (5). What is your next strategic move?"
prompt = "You are a negotiation agent. Current state:\n\n" + negotiation_state + "\n\nYour move (one short paragraph):"

tok_infer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tok_infer.pad_token = tok_infer.eos_token
inp = tok_infer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

base_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to(device)
trained_path = PHASE2_OUTPUT if os.path.isdir(PHASE2_OUTPUT) else PHASE1_OUTPUT
trained_model = AutoModelForCausalLM.from_pretrained(trained_path).to(device)

with torch.no_grad():
    out_base = base_model.generate(**inp, max_new_tokens=60, do_sample=True, temperature=0.7, pad_token_id=tok_infer.eos_token_id)
    out_trained = trained_model.generate(**inp, max_new_tokens=60, do_sample=True, temperature=0.7, pad_token_id=tok_infer.eos_token_id)

dec_base = tok_infer.decode(out_base[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
dec_trained = tok_infer.decode(out_trained[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("=== Base TinyLlama ===")
print(dec_base)
print()
print("=== Trained (Phase 1+2) ===")
print(dec_trained)

In [ ]:
# Run BluffDetector on the camera bluff message — show all 4 signals
from simulation.seller_profiles import get_profile
from simulation.seller_sim import CraigslistSellerSim
from agent.bluff_detector import analyze_from_sim

profile = get_profile("seller_bluffer_camera")
seller = CraigslistSellerSim(profile)
messages = ["Hi, interested in the camera. Would you take $38?", "How about $32?", "Come on, can you do $30?",]
last_response = None
for msg in messages:
    last_response = seller.step(msg)

if last_response:
    signals = analyze_from_sim(seller, last_response)
    print("Camera bluff message:", repr(profile.get("bluff_message", "")))
    print()
    print("BluffDetector — all 4 signals:")
    print("  timing_tell    =", signals.timing_tell)
    print("  size_tell      =", signals.size_tell)
    print("  formulaic_tell =", signals.formulaic_tell)
    print("  pattern_tell   =", signals.pattern_tell)
    print("  bluff_score    =", signals.bluff_score)
    print("  is_bluff       =", signals.is_bluff)
else:
    print("No seller response (ghosted).")

## Summary — Tracks Hit & Submission

| Track | How ArbitrAgent hits it |
|-------|-------------------------|
| **Multi-Agent** | Agent manages 9–12 simultaneous counterpart LLMs (sellers + trade targets) |
| **Long-Horizon** | Route-confirmation arc spans multiple rounds with full state tracking |
| **Self-Improvement** | Curriculum RL: Phase 1 (Diplomacy) + Phase 2 (Human Imitation), measurable reward improvement |
| **Wild Card** | Autonomous capital deployment via confirmed route arbitrage ($20 → execute) |
| **Halluminate $10k** | Agent managing multiple actors to discover and achieve the task |
| **Fleet AI $10k** | Bluff detection layer as oversight agent scoring counterpart behavior |

**Submission links:**
- Repo: [GitHub](https://github.com/your-username/Play-gent)
- Demo: [HuggingFace Spaces](https://huggingface.co/spaces/your-username/arbitragent)
- Video: [1-min YouTube](https://youtube.com/...)
- Submit: [cerebralvalley.ai](https://cerebralvalley.ai) — Sunday 1:00 PM